# Before the workshop: check your setup

Before typing commands, here are the tools you will use:

- **Python environment:** the workshop’s own Python and installed packages, stored in `.venv` so they stay together.
- **Terminal / PowerShell:** the command window where you install packages and start JupyterLab.
- **JupyterLab:** the application that opens a notebook in your browser. A notebook contains explanations and code cells.
- **Kernel:** the running Python process that executes those cells and remembers their variables. Select **Python (napari workshop)**. Restarting it clears those variables.
- **Napari:** the separate desktop window where you view and manipulate images and layers.

Keep the terminal running while you use JupyterLab and napari. More terms are in the [glossary](../handouts/glossary.md).

Allow 10 minutes after installation. Select **Python (napari workshop)** in Jupyter's kernel selector. Click a code cell and press **Shift+Enter** to run it. A number appears beside a completed cell; `[*]` means it is still working.

You will use three windows: the terminal starts Jupyter; your browser contains these instructions; napari opens as a separate desktop window. Keep the terminal open. Complete the [installation instructions](../setup/windows.md) or [Mac instructions](../setup/macos.md) first.

This notebook checks your selected kernel. A standalone script run by uv can have its own environment, so its success alone does not identify the environment selected here.

In [ ]:
import sys
import platform
import importlib.metadata as packages
print("Python:", sys.executable)
print("Computer:", platform.system(), platform.machine())
for name in ("napari", "numpy", "ipykernel", "imageio-ffmpeg"):
    print(name, packages.version(name))

**Check:** the Python path should contain the workshop environment (`.venv` on your laptop). This release uses napari 0.6.6 and Python 3.12. If an import fails, check the kernel before reinstalling anything. `import` makes an installed package available in this running notebook.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data" / "zenodo-manifest.json").is_file())
DATA = ROOT / "data" / "teaching"
print(DATA)

The `/` symbol above joins folder names portably on Windows and Mac. The helper finds the workshop folder from your current notebook location. Download and extract the teaching data before continuing. The 42.8 MB [teaching dataset](https://doi.org/10.5281/zenodo.22696913) is a spatially reduced derivative of the [original dataset](https://doi.org/10.5281/zenodo.22694247).

The next cell checks every image header, reads and validates both tables, and verifies all file checksums recorded in the teaching metadata. It does not load the full image series into RAM. Any error means the data are not ready; follow the [data recovery instructions](../setup/troubleshooting.md).

In [ ]:
from napari_workshop.data import load_images, load_tables
from napari_workshop.download import verified

# Validate every TIFF header and both tables without loading the image series.
images, image_scale, metadata = load_images(DATA, lazy=True)
detections, tracks = load_tables(DATA, metadata)
for spec in metadata["files"]:
    if not verified(DATA / spec["name"], spec):
        raise ValueError(f"Checksum verification failed: {spec['name']}. See data recovery instructions.")
print("Data verified:", len(metadata["files"]), "files; both channels and tables are ready.")

## Check graphics with a tiny practice object
`%gui qt` lets the notebook and napari's window respond at the same time. The small array below contains a bright sphere. This practice image is synthetic, not an experimental result. Run the cell, find the napari window, and drag to rotate the sphere. **Pause here to inspect the window; do not use Run All for this interactive check.**

The sphere contains values 0 and 1. Explicit contrast limits map 0 to black and 1 to white; the default 0–255 display range for this image type would make it almost invisible.

In [ ]:
%gui qt
import numpy as np
import napari
z, y, x = np.indices((32, 32, 32))
sphere = ((z - 16)**2 + (y - 16)**2 + (x - 16)**2 < 100).astype("uint8")
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(sphere, name="Practice sphere", contrast_limits=(0, 1))

**Check:** a sphere appears and rotates. A successful import with an empty/black/broken window is not a passed graphics check. See [troubleshooting](../setup/troubleshooting.md). Close the practice viewer with the next cell before the course; closing napari does not close Jupyter.

In [ ]:
viewer.close()

## Check movie encoding in this kernel
The command below starts the check script using **this notebook's Python executable**. It writes and reads three tiny movie frames in a temporary folder. No large image is loaded and no system-wide FFmpeg installation is required if the packaged encoder works.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "scripts" / "check_setup.py"), "--movie"], check=True)

**Ready when:** the correct Python path is shown, the complete data verification succeeds, the sphere rotates, and the movie check succeeds. Write down your OS, processor type, RAM, and any problem before the workshop. Windows and macOS instructions need a real pilot on those platforms; the included validation report records which tests have actually been performed.